In [7]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test



In [2]:


#subj = sys.argv[1] ## name of participant list
# Variables path
layer_script = "block"

#process=""
disco="g"

# Carpeta general
datadir = Path(f"{disco}:\MOUS_204")

#carpetas generales de datos
# mri_dir = datadir / f"{subj}"/"anat"
# meg_dir = datadir / f"{subj}"/"meg"

# Carpeta de preprocesado
output_preproc = datadir / "output_preproc"

channels_structure_path = output_preproc / "channels_structure"


preproc_path = output_preproc / f"preproc_{layer_script}"
preproc_path.mkdir(parents=True, exist_ok=True)
 
# Carpeta de epocas "sucias"
epochs_path = preproc_path / f"epochs_{layer_script}"
epochs_path.mkdir(parents=True, exist_ok=True) 

# Carpeta de ICA
ICA_path = preproc_path / f"ICA_{layer_script}"
ICA_path.mkdir(parents=True, exist_ok=True)

# Épocas limpias
epochs_clean_path = preproc_path / f"epochs_clean_{layer_script}"
epochs_clean_path.mkdir(parents=True, exist_ok=True)

#epocas evoked
evoked_path = Path(preproc_path) / f"evoked_{layer_script}"
evoked_path.mkdir(parents=True, exist_ok=True)
 

# Definir la carpeta de output_source antes de usarla
output_source = Path(r"g:\MOUS_204\output_source")

source_path = output_source / f"source_{layer_script}"
source_path.mkdir(parents=True, exist_ok=True)

#raw_hsp es el raw con fiducials cargados
raw_hsp_path = source_path / f"raw_hsp"
raw_hsp_path.mkdir(parents=True, exist_ok=True)

# Carpeta de forward solution
fwd_path = source_path / f"fwd"
fwd_path.mkdir(parents=True, exist_ok=True)

# Carpeta de inverse solution
inverse_path = source_path / f"inverse"
inverse_path.mkdir(parents=True, exist_ok=True)

output_analysis = datadir / "output_analysis"
analysis_path = output_analysis / f"analysis_{layer_script}"
analysis_path.mkdir(parents=True, exist_ok=True)

ACW_path = analysis_path / f"acw_{layer_script}"
ACW_path.mkdir(parents=True, exist_ok=True)

PLE_path = analysis_path / f"PLE_{layer_script}"
PLE_path.mkdir(parents=True, exist_ok=True)

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

<>:9: SyntaxWarning: invalid escape sequence '\M'
<>:9: SyntaxWarning: invalid escape sequence '\M'
C:\Users\UCM\AppData\Local\Temp\ipykernel_14104\3531330929.py:9: SyntaxWarning: invalid escape sequence '\M'
  datadir = Path(f"{disco}:\MOUS_204")


In [ ]:
# acw_all = pd.read_pickle(ACW_path /f"autocorrelation_subjects_all.pickle")

#lectura de acw
acw_50_df = pd.read_pickle(ACW_path /f"acw_50_df.pickle")


##creacion dataframe ACW_50

# acw_50_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_50_elect_all_epoch_all"]]
# acw_50_df.to_pickle(ACW_path / f"acw_50_df.pickle")

# acw_0_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_0_elect_all_epoch_all"]]
# acw_0_df.to_pickle(ACW_path / f"acw_0_df.pickle")

# del acw_all, del acw_0_df

In [4]:
channels = pd.read_csv(channels_structure_path / "channels_mag.csv")
channels_mag=channels[channels["canal_efectivo"].notna()]["canal_efectivo"]

channels_mag=channels_mag.tolist()
print(channels_mag)
indice_channels_efectivos = channels[channels["canal_efectivo"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels

['MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF62-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

In [5]:
##valores de las columnas
condition = acw_50_df["Condition"].unique()
print("Condiciones en los datos:", condition)

subjects = acw_50_df["Subject"].unique()
print("Sujetos en los datos:", subjects)

elect_all= acw_50_df["Elect"].unique()
print("sensores en los datos:", elect_all)

epochs_all= acw_50_df["Epoch"].unique()
print("Epochs en los datos:", epochs_all)


Condiciones en los datos: ['zinnen' 'woorden']
Sujetos en los datos: ['sub-A2002' 'sub-A2003' 'sub-A2004' 'sub-A2005' 'sub-A2006' 'sub-A2007'
 'sub-A2008' 'sub-A2009' 'sub-A2010' 'sub-A2013' 'sub-A2014' 'sub-A2015'
 'sub-A2016' 'sub-A2017' 'sub-A2019' 'sub-A2020' 'sub-A2021' 'sub-A2024'
 'sub-A2025']
sensores en los datos: ['MLC12-4304' 'MLC13-4304' 'MLC14-4304' 'MLC15-4304' 'MLC16-4304'
 'MLC17-4304' 'MLC21-4304' 'MLC22-4304' 'MLC23-4304' 'MLC24-4304'
 'MLC25-4304' 'MLC31-4304' 'MLC32-4304' 'MLC41-4304' 'MLC42-4304'
 'MLC51-4304' 'MLC52-4304' 'MLC53-4304' 'MLC54-4304' 'MLC55-4304'
 'MLC61-4304' 'MLC62-4304' 'MLC63-4304' 'MLF11-4304' 'MLF12-4304'
 'MLF13-4304' 'MLF14-4304' 'MLF21-4304' 'MLF22-4304' 'MLF23-4304'
 'MLF24-4304' 'MLF25-4304' 'MLF31-4304' 'MLF32-4304' 'MLF33-4304'
 'MLF34-4304' 'MLF35-4304' 'MLF41-4304' 'MLF42-4304' 'MLF43-4304'
 'MLF44-4304' 'MLF45-4304' 'MLF46-4304' 'MLF51-4304' 'MLF52-4304'
 'MLF53-4304' 'MLF54-4304' 'MLF55-4304' 'MLF56-4304' 'MLF61-4304'
 'MLF62-4304' '

In [ ]:
## get info 

#read epochs to build evoked
epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{"sub-A2002"}_epochs_zinnen_block-epo.fif")
len(epochs_zinnen.pick("mag", exclude="bads").ch_names)

evokeds_zinnen=epochs_zinnen.average()
## evokeds_zinnen es una lista

##cojo el primer elemento, solo tengo una lista
evoked_zinnen=evokeds_zinnen

info=evoked_zinnen.info
del epochs_zinnen
del evokeds_zinnen
del evoked_zinnen





Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2002_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
17 matching events found
No baseline correction applied
0 projection items activated


# General differneces between woorden and zinnen conditions

In [8]:
##selección de valores
##matrices de valores de acw_50, divididas por condicion y tipo de channel
X_zinnen = []

X_woorden = []


##Filtras la condicion
for cond in condition:
    acw_50_condition_df=acw_50_df[acw_50_df["Condition"] == f"{cond}"]


#         #filtras por sujeto 
    for subj in subjects:
    #creacion de lista de valores de acw_50 para cada sujeto
        acw_50_epoch_list= []
#       #filtras por sujeto
        for epoch in epochs_all:
            # Extraer el valor de ACW_50 para la combinación actual
            try:
            #coges el valor de acw_50 para el sujeto y el epoch
                print(f"subj:{subj},epoch, {epoch}")
                acw_50_elect_all_epoch_all = acw_50_condition_df[(acw_50_condition_df["Subject"] == subj) & (acw_50_condition_df["Epoch"] == epoch)]["acw_50_elect_all_epoch_all"]
                if not acw_50_elect_all_epoch_all.empty:
                    acw_50_epoch_list.append(acw_50_elect_all_epoch_all)

            except Exception as e:
                print(e, "probablemente faltaban epochs en algunos sujetos")
                continue
        # Convertir la lista a un array de numpy
        acw_50_epoch_array = np.array(acw_50_epoch_list)
        #take the mean on epochs
        acw_50_epoch_mean = np.mean(acw_50_epoch_array, axis=0)

        # #add it to the different lists
        if cond == "zinnen":
            X_zinnen.append(acw_50_epoch_mean)

        if cond == "woorden":
            X_woorden.append(acw_50_epoch_mean)


X_zinnen = np.array(X_zinnen)
X_woorden = np.array(X_woorden)

##take the mean on channels
X_zinnen_mean = np.mean(X_zinnen, axis=1)
X_woorden_mean = np.mean(X_woorden, axis=1)



print("Shapes de las matrices de valores ACW_50:\n")

print("▶ Condición: ZINNEN")
print("Condition zinnen:", X_zinnen_mean.shape)
print("Condition woorden:", X_woorden_mean.shape)


subj:sub-A2002,epoch, 0
subj:sub-A2002,epoch, 1
subj:sub-A2002,epoch, 2
subj:sub-A2002,epoch, 3
subj:sub-A2002,epoch, 4
subj:sub-A2002,epoch, 5
subj:sub-A2002,epoch, 6
subj:sub-A2002,epoch, 7
subj:sub-A2002,epoch, 8
subj:sub-A2002,epoch, 9
subj:sub-A2002,epoch, 10
subj:sub-A2002,epoch, 11
subj:sub-A2002,epoch, 12
subj:sub-A2002,epoch, 13
subj:sub-A2002,epoch, 14
subj:sub-A2002,epoch, 15
subj:sub-A2002,epoch, 16
subj:sub-A2002,epoch, 17
subj:sub-A2002,epoch, 18
subj:sub-A2002,epoch, 19
subj:sub-A2002,epoch, 20
subj:sub-A2002,epoch, 21
subj:sub-A2002,epoch, 22
subj:sub-A2002,epoch, 23
subj:sub-A2003,epoch, 0
subj:sub-A2003,epoch, 1
subj:sub-A2003,epoch, 2
subj:sub-A2003,epoch, 3
subj:sub-A2003,epoch, 4
subj:sub-A2003,epoch, 5
subj:sub-A2003,epoch, 6
subj:sub-A2003,epoch, 7
subj:sub-A2003,epoch, 8
subj:sub-A2003,epoch, 9
subj:sub-A2003,epoch, 10
subj:sub-A2003,epoch, 11
subj:sub-A2003,epoch, 12
subj:sub-A2003,epoch, 13
subj:sub-A2003,epoch, 14
subj:sub-A2003,epoch, 15
subj:sub-A2003,epoch

In [9]:


##dependent condition
def paired_statistic(diff, _):
    return np.mean(diff)

# Crear el diccionario con nombres descriptivos
data_dict = {
    "X_zinnen_mean": X_zinnen_mean,
    "X_woorden_mean": X_woorden_mean}



print("DEPENDENT analysis")

print(f"for differences between zinnen and woorden")

x= data_dict[f"X_zinnen_mean"]
##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
y= data_dict[f"X_woorden_mean"]


diff=x-y
# Permutation test
res = permutation_test(
    (diff, np.zeros_like(diff)),
    #
    statistic=paired_statistic,
    vectorized=False,
    n_resamples=10000,
    alternative='greater', 
    random_state=42
)

print(f"Statistic value: {res.statistic}")
print(f"p-value: {res.pvalue}")

DEPENDENT analysis
for differences between zinnen and woorden
Statistic value: 0.00040322629315829274
p-value: 0.0004999500049995


# Cluster analysis

## Datos a comparar

ahora tengo por cada condición: subject x epoch x channel

#### procedimiento

- promediar a nivel de epoca para tener subject x channel (luego probaré a hacerlo de otra manera sin promediar)

- ejecutar permutation cluster based analysis usando mi acw_zinnen vs mi acw_woorden  que va a estar como subject x channel

    - el cluster solo puede ser espacial, porque mis epocas son discontinuas, creo que esto es adjacency 

#### Resultados

T_obs: La estadística observada en cada punto.
clusters: Los clústeres formados por puntos significativos adyacentes.
cluster_p_values: Los valores p de cada clúster (ya corregidos por comparaciones múltiples).
H0: La distribución nula obtenida por permutaciones.



Supongo que clusters y sus valores de significacioón
Luego no se como se compara entre condiciones, pero de momemento vamos así

## Cluster in differences

In [80]:
##get adjacency value


adjacency_reduced =  pd.read_pickle(channels_structure_path /f"adjacency_reduced.pkl")
print(f"shape adjacency_reduced: {adjacency_reduced.shape}")

shape adjacency_reduced: (272, 272)


In [10]:
X_zinnen = []
X_woorden = []

for cond in ["zinnen", "woorden"]:
    for subj  in subjects:
            acw_50_epoch_list= []
            for epoch in epochs_all:

                # Extraer el valor de ACW_50 para la combinación actual
                # try:
                print(subj, cond, epoch)

                acw_50_elect_all_epoch_all = acw_50_df[(acw_50_df["Subject"] == subj) & (acw_50_df["Condition"] == cond) & (acw_50_df["Epoch"] == epoch)]["acw_50_elect_all_epoch_all"]
                if not acw_50_elect_all_epoch_all.empty:
                    acw_50_epoch_list.append(acw_50_elect_all_epoch_all)

                    

                # except Exception as e:
                #     print(e, "probablemente faltaban epochs en algunos sujetos")
                #     continue
            acw_50_epoch_array = np.array(acw_50_epoch_list)
            # average over epochs
            acw_50_epoch_mean = np.mean(acw_50_epoch_array, axis=0)

            if cond == "zinnen":
                X_zinnen.append(acw_50_epoch_mean)
            if cond == "woorden":
                X_woorden.append(acw_50_epoch_mean)

X_zinnen = np.array(X_zinnen)
X_woorden = np.array(X_woorden)

print("Shapes of matrix ACW_50:\n")
print("Condition zinnen:", X_zinnen.shape)
print("Condition woorden:", X_woorden.shape)

sub-A2002 zinnen 0
sub-A2002 zinnen 1
sub-A2002 zinnen 2
sub-A2002 zinnen 3
sub-A2002 zinnen 4
sub-A2002 zinnen 5
sub-A2002 zinnen 6
sub-A2002 zinnen 7
sub-A2002 zinnen 8
sub-A2002 zinnen 9
sub-A2002 zinnen 10
sub-A2002 zinnen 11
sub-A2002 zinnen 12
sub-A2002 zinnen 13
sub-A2002 zinnen 14
sub-A2002 zinnen 15
sub-A2002 zinnen 16
sub-A2002 zinnen 17
sub-A2002 zinnen 18
sub-A2002 zinnen 19
sub-A2002 zinnen 20
sub-A2002 zinnen 21
sub-A2002 zinnen 22
sub-A2002 zinnen 23
sub-A2003 zinnen 0
sub-A2003 zinnen 1
sub-A2003 zinnen 2
sub-A2003 zinnen 3
sub-A2003 zinnen 4
sub-A2003 zinnen 5
sub-A2003 zinnen 6
sub-A2003 zinnen 7
sub-A2003 zinnen 8
sub-A2003 zinnen 9
sub-A2003 zinnen 10
sub-A2003 zinnen 11
sub-A2003 zinnen 12
sub-A2003 zinnen 13
sub-A2003 zinnen 14
sub-A2003 zinnen 15
sub-A2003 zinnen 16
sub-A2003 zinnen 17
sub-A2003 zinnen 18
sub-A2003 zinnen 19
sub-A2003 zinnen 20
sub-A2003 zinnen 21
sub-A2003 zinnen 22
sub-A2003 zinnen 23
sub-A2004 zinnen 0
sub-A2004 zinnen 1
sub-A2004 zinnen 2
sub

In [ ]:
diff= X_zinnen - X_woorden
print(f"diff.shape is {diff.shape}")



# as I´m going to apply differences to same subjects, i need a DEPENDENT t test kind of thing, 
# so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
#                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# as im going to put zinnen before woorden i need tail=1

#instead of puting X with bot condigions, i will put the difference between them diff=zinnen-woorden


t_obs_diff, clusters_diff, clusters_pv_diff, H0=mne.stats.permutation_cluster_1samp_test(diff, threshold=None, n_permutations=1024, tail=1, 
                                                                          stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
                                                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# Obtenemos los clusters como listas de índices
# Filtrar clusters con p < 0.05
#zip links clusters y p-values
significant_clusters_diff = [
    cluster for cluster, p in zip(clusters_diff, clusters_pv_diff) if p < 0.05
]

print(f"{len(significant_clusters_diff)} significative clusters found.")
print(f"Clusters significative: {significant_clusters_diff}")

diff.shape is (19, 272)
shape adjacency_reduced: (272, 272)
Using a threshold of 1.734064
stat_fun(H1): min=-1.1705784353109319 max=4.1363866876308
Running initial clustering …
Found 4 clusters


100%|██████████| Permuting : 1023/1023 [00:00<00:00, 5133.57it/s]

1 significative clusters found.
Clusters significative: [(array([  0,   1,   2,   3,   7,   8,  11,  12,  13,  14,  15,  17,  18,
        21,  22,  23,  27,  28,  29,  30,  32,  33,  34,  35,  36,  37,
        38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  51,
        52,  53,  79,  84,  97,  98, 135, 148, 152, 153, 154, 161, 162,
       164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176,
       177, 178, 179, 180, 184, 185, 263, 266, 267]),)]


In [91]:
#read epochs to build evoked
epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{"sub-A2002"}_epochs_zinnen_block-epo.fif")
len(epochs_zinnen.pick("mag", exclude="bads").ch_names)

evokeds_zinnen=epochs_zinnen.average()
## evokeds_zinnen es una lista

##cojo el primer elemento, solo tengo una lista
evoked_zinnen=evokeds_zinnen

info=evoked_zinnen.info
del epochs_zinnen
del evokeds_zinnen
del evoked_zinnen

mask_diff = np.zeros(t_obs_diff.shape, dtype=bool)

##significant_clusters_diff is a list of tuples of arrays
# cluster[0] is the electrodes of the  cluster
for cluster in significant_clusters_diff:
    mask_diff[cluster[0]] = True 



##in data i use the value of t_obs_diff
#pos takes the information from evoked.info object
#mask takes the mask_diff object
#in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric test,
# even if the distribution is non parametric 

ax, im=mne.viz.plot_topomap(
    data=t_obs_diff,
    pos=info,
    mask=mask_diff,
    cmap='RdBu_r',
    vlim=(-np.max(np.abs(t_obs_diff)), np.max(np.abs(t_obs_diff))),
    mask_params=dict(marker='o', markerfacecolor='yellow', markersize=10),
    contours=0,
    show=True
)

fig=ax.get_figure()
fig.suptitle("Clusters in differences", fontsize=14)

Reading g:\MOUS_204\output_preproc\preproc_block\epochs_clean_block\sub-A2002_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
17 matching events found
No baseline correction applied
0 projection items activated


Text(0.5, 0.98, 'Clusters in differences')

## plot topomap for word condition

In [90]:

# as I´m going to apply woordenerences to same subjects, i need a DEPENDENT t test kind of thing, 
# so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
#                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# as im going to put zinnen before woorden i need tail=1

#instead of puting X with bot condigions, i will put the woordenerence between them woorden=zinnen-woorden


t_obs_woorden, clusters_woorden, clusters_pv_woorden, H0=mne.stats.permutation_cluster_1samp_test(X_woorden, threshold=None, n_permutations=1024, tail=1, 
                                                                          stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
                                                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# Obtenemos los clusters como listas de índices
# Filtrar clusters con p < 0.05
#zip links clusters y p-values
significant_clusters_woorden = [
    cluster for cluster, p in zip(clusters_woorden, clusters_pv_woorden) if p < 0.05
]

print(f"{len(significant_clusters_woorden)} significative clusters found.")
print(f"Clusters significative: {significant_clusters_woorden}")

mask_woorden = np.zeros(t_obs_woorden.shape, dtype=bool)

##significant_clusters_woorden is a list of tuples of arrays
# cluster[0] is the electrodes of the  cluster
for cluster in significant_clusters_woorden:
    mask_woorden[cluster[0]] = True 



##in data i use the value of t_obs_woorden
#pos takes the information from evoked.info object
#mask takes the mask_woorden object
#in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric test,
# even if the distribution is non parametric 

ax, im=mne.viz.plot_topomap(
    data=t_obs_woorden,
    pos=info,
    mask=mask_woorden,
    cmap='RdBu_r',
    vlim=(-np.max(np.abs(t_obs_woorden)), np.max(np.abs(t_obs_woorden))),
    mask_params=dict(marker='o', markerfacecolor='yellow', markersize=10),
    contours=0,
    show=True
)
fig = ax.get_figure()
fig.suptitle("clusters in woorden", fontsize=14)

Using a threshold of 1.734064
stat_fun(H1): min=4.1186119114373385 max=40.3026669960027
Running initial clustering …
Found 1 cluster


100%|██████████| Permuting : 1023/1023 [00:00<00:00, 10869.10it/s]

1 significative clusters found.
Clusters significative: [(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
      

Text(0.5, 0.98, 'clusters in woorden')

## permutation cluster for zinnen

In [ ]:


# as I´m going to apply zinnenerences to same subjects, i need a DEPENDENT t test kind of thing, 
# so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
#                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# as im going to put zinnen before zinnen i need tail=1

#instead of puting X with bot condigions, i will put the zinnenerence between them zinnen=zinnen-zinnen


t_obs_zinnen, clusters_zinnen, clusters_pv_zinnen, H0=mne.stats.permutation_cluster_1samp_test(X_zinnen, threshold=None, n_permutations=1024, tail=0, 
                                                                          stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
                                                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# Obtenemos los clusters como listas de índices
# Filtrar clusters con p < 0.05
#zip links clusters y p-values
significant_clusters_zinnen = [
    cluster for cluster, p in zip(clusters_zinnen, clusters_pv_zinnen) if p < 0.05
]

print(f"{len(significant_clusters_zinnen)} significative clusters found.")
print(f"Clusters significative: {significant_clusters_zinnen}")
#read epochs to build evoked
epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{"sub-A2002"}_epochs_zinnen_block-epo.fif")
len(epochs_zinnen.pick("mag", exclude="bads").ch_names)

evokeds_zinnen=epochs_zinnen.average()
## evokeds_zinnen es una lista

##cojo el primer elemento, solo tengo una lista
evoked_zinnen=evokeds_zinnen

info=evoked_zinnen.info
del epochs_zinnen
del evokeds_zinnen
del evoked_zinnen

mask_zinnen = np.zeros(t_obs_zinnen.shape, dtype=bool)

##significant_clusters_zinnen is a list of tuples of arrays
# cluster[0] is the electrodes of the  cluster
for cluster in significant_clusters_zinnen:
    mask_zinnen[cluster[0]] = True 



##in data i use the value of t_obs_zinnen
#pos takes the information from evoked.info object
#mask takes the mask_zinnen object
#in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric test,
# even if the distribution is non parametric 
ax,im= mne.viz.plot_topomap(
    data=t_obs_zinnen,
    pos=info,
    mask=mask_zinnen,
    cmap='RdBu_r',
    vlim=(-np.max(np.abs(t_obs_zinnen)), np.max(np.abs(t_obs_zinnen))),
    mask_params=dict(marker='o', markerfacecolor='yellow', markersize=10),
    contours=0,
    show=True
)

fig = ax.get_figure()
fig.suptitle("clusters in zinnen", fontsize=14)


Using a threshold of 2.100922
stat_fun(H1): min=2.8469583094537425 max=38.963997765365214
Running initial clustering …
Found 1 cluster


100%|██████████| Permuting : 1023/1023 [00:00<00:00, 11415.85it/s]

1 significative clusters found.
Clusters significative: [(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
      

    Found the data of interest:
        t =       0.00 ...   38000.00 ms
        0 CTF compensation matrices available
Not setting metadata
17 matching events found
No baseline correction applied
0 projection items activated


Text(0.5, 0.98, 't_obs_zinnen')